# EX — Classical ML Advanced Real-World Exercises

SVM, Naive Bayes, Gradient Boosting, PCA, hyperparameter tuning, and a classical
time-series baseline. Requires `scikit-learn`, `statsmodels`.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=600, n_features=10, n_informative=6, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1. SVM — Maximum Margin Classification
**Pointer:** always scale features for SVM.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm = SVC(kernel="rbf", C=1.0)
svm.fit(X_train_s, y_train)
print("SVM accuracy:", accuracy_score(y_test, svm.predict(X_test_s)))


## 2. Naive Bayes — Fast Probabilistic Baseline

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train, y_train)  # no scaling needed
print("Naive Bayes accuracy:", accuracy_score(y_test, nb.predict(X_test)))


## 3. Gradient Boosting — The Strongest Tabular Baseline

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=0)
gb.fit(X_train, y_train)
print("Gradient Boosting accuracy:", accuracy_score(y_test, gb.predict(X_test)))


### TODO 1
Compare all three models' accuracy in a small table (dict or DataFrame) and identify the best one.

In [ ]:
# TODO
results = {}
print(results)


<details><summary>Solution</summary>

```python
results = {
  'SVM': accuracy_score(y_test, svm.predict(X_test_s)),
  'NaiveBayes': accuracy_score(y_test, nb.predict(X_test)),
  'GradientBoosting': accuracy_score(y_test, gb.predict(X_test)),
}
best = max(results, key=results.get)
```
</details>

## 4. PCA — Dimensionality Reduction for Visualization

In [ ]:
from sklearn.decomposition import PCA
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_train_s)
print("explained variance ratio:", pca.explained_variance_ratio_)

plt.figure(figsize=(5,4))
plt.scatter(X_2d[:,0], X_2d[:,1], c=y_train, cmap="coolwarm", alpha=0.6)
plt.title("PCA projection to 2D")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.savefig("/tmp/pca_plot.png", dpi=80, bbox_inches="tight")
plt.close()
print("saved plot")


### TODO 2
How many PCA components are needed to explain at least 90% of the variance? (hint: `np.cumsum` on `explained_variance_ratio_` from a PCA fit with more components).

In [ ]:
# TODO
pca_full = PCA().fit(X_train_s)
n_components_90 = None
print(n_components_90)


<details><summary>Solution</summary>

```python
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90 = int(np.argmax(cumulative >= 0.9) + 1)
```
</details>

## 5. Hyperparameter Tuning with GridSearchCV
**Pointer:** never tune against your held-out test set — GridSearchCV does cross-validation internally on the training set.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"n_estimators": [50, 100], "max_depth": [2, 3]}
grid = GridSearchCV(GradientBoostingClassifier(random_state=0), param_grid, cv=3, scoring="accuracy")
grid.fit(X_train, y_train)
print("best params:", grid.best_params_)
print("test accuracy with best params:", accuracy_score(y_test, grid.best_estimator_.predict(X_test)))


## 6. Classical Time-Series Forecasting

In [ ]:
import pandas as pd
np.random.seed(0)
dates = pd.date_range("2023-01-01", periods=100, freq="D")
trend = np.linspace(50, 80, 100)
noise = np.random.normal(0, 3, 100)
series = pd.Series(trend + noise, index=dates)

# Simple moving average baseline
ma_forecast = series.rolling(window=7).mean()
print(ma_forecast.tail())


### TODO 3
Fit a simple ARIMA(1,1,1) model on `series` using `statsmodels`, forecast the next 5 days, and compare visually/numerically to the moving-average approach.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# TODO
model = None
forecast = None
print(forecast)


<details><summary>Solution</summary>

```python
model = ARIMA(series, order=(1,1,1)).fit()
forecast = model.forecast(steps=5)
```
</details>


## Key Takeaways
- SVM needs scaled features; Naive Bayes and tree-based models generally don't.
- Gradient boosting is usually the strongest tabular baseline, but needs careful tuning.
- PCA's explained_variance_ratio_ tells you how much information you keep as you reduce dimensions.
- Always tune hyperparameters via cross-validation (GridSearchCV), never against the final test set.
- Check stationarity and consider differencing before fitting ARIMA models.
